# Product Analytics using Python

In [ ]:
# Loading dataset
import pandas as pd
pd.set_option('display.max_columns', None)
df = pd.read_excel('../../data/raw/Online Retail.xlsx', sheet_name='Online Retail')
df.shape

In [ ]:
df.head()

In [ ]:
import janitor
df = df.clean_names(case_type='snake')

In [ ]:
df.head()

In [ ]:
df['quantity'].plot.box(showfliers=False, grid=True, figsize=(10, 5))

In [ ]:
# Screen out data Quantity > 0
df = df[df['quantity'] > 0]

In [ ]:
df['quantity'].plot.box(showfliers=False, grid=True, figsize=(10, 5))

# Visualize time series trend

In [ ]:
monthly_orders_df = df.set_index('invoice_date')['invoice_no'].resample('ME').nunique()

In [ ]:
monthly_orders_df

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=[x.strftime('%m.%Y') for x in monthly_orders_df.index],
    y=monthly_orders_df.values,
    mode='lines+markers'
))

fig.update_layout(
    title='Total Number of Orders Over Time',
    xaxis_title='date',
    yaxis_title='number of orders/invoices',
    xaxis=dict(tickangle=45),
    width=1000,
    height=700,
    template='ggplot2'
)

fig.show()

In [ ]:
invoice_dates = df.loc[
    df['invoice_date'] >= '2011-12-01',
    'invoice_date'
]

In [ ]:
print(f"Minimum invoice date: {invoice_dates.min()}")
print(f"Maximum invoice date: {invoice_dates.max()}")

In [ ]:
# Filtering out data with invoice date < 2011-12-01
df = df[df['invoice_date'] < '2011-12-01']

In [ ]:
# Create monthly orders dataframe
monthly_orders_df = df.set_index('invoice_date')['invoice_no'].resample('ME').nunique()
monthly_orders_df

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=[x.strftime('%m.%Y') for x in monthly_orders_df.index],
    y=monthly_orders_df.values,
    mode='lines+markers'
))

fig.update_layout(
    title='Total Number of Orders Over Time',
    xaxis_title='date',
    yaxis_title='number of orders/invoices',
    xaxis=dict(tickangle=45),
    width=1000,
    height=700,
    template='ggplot2'
)

fig.show()

In [ ]:
# Create monthly revenues dataframe
df['revenue'] = df['quantity'] * df['unit_price']
monthly_revenues_df = df.set_index('invoice_date')['revenue'].resample('ME').sum()
monthly_revenues_df

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=[x.strftime('%m.%Y') for x in monthly_revenues_df.index],
    y=monthly_revenues_df.values,
    mode='lines+markers',
    name='sales'
))

fig.update_layout(
    title='Total Revenue Over Time',
    xaxis_title='date',
    yaxis_title='sales',
    xaxis=dict(tickangle=45),
    yaxis=dict(range=[0, monthly_revenues_df.max() + 100000]),
    width=1000,
    height=700,
    template='ggplot2'
)

fig.show()

# Repeate Customer Behavior

In [ ]:
# Create invoice_customer_df
invoice_customer_df = df.groupby(
    by=['invoice_no', 'invoice_date']
).agg({
    'revenue': sum,
    'customer_id': max,
    'country': max,
}).reset_index()

invoice_customer_df.head()

In [ ]:
monthly_repeat_customers_df = invoice_customer_df.set_index('invoice_date').groupby([
                                                            pd.Grouper(freq='ME'), 'customer_id'
                                                            ]).filter(lambda x: len(x) > 1).resample('ME').nunique()['customer_id']
monthly_repeat_customers_df

In [ ]:
monthly_unique_customers_df = df.set_index('invoice_date')['customer_id'].resample('ME').nunique()
monthly_unique_customers_df

In [ ]:
monthly_repeat_percentage = monthly_repeat_customers_df/monthly_unique_customers_df*100.0
monthly_repeat_percentage

In [ ]:
x_vals = monthly_repeat_customers_df.index

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x_vals,
    y=monthly_repeat_customers_df.values,
    mode='lines+markers',
    name='Repeat Customers'
))

fig.add_trace(go.Scatter(
    x=x_vals,
    y=monthly_unique_customers_df.values,
    mode='lines+markers',
    name='All Customers'
))

fig.add_trace(go.Bar(
    x=x_vals,
    y=monthly_repeat_percentage.values,
    name='Percentage of Repeat',
    marker_color='green',
    opacity=0.2,
    yaxis='y2'
))

fig.update_layout(
    title='Number of All vs. Repeat Customers Over Time',
    xaxis=dict(
        title='date',
        tickformat='%m.%Y',
        tickangle=45
    ),
    yaxis=dict(
        title='number of customers',
        range=[0, monthly_unique_customers_df.max() + 100]
    ),
    yaxis2=dict(
        title='percentage (%)',
        overlaying='y',
        side='right',
        range=[0, 100]
    ),
    width=1000,
    height=700,
    template='ggplot2',
    legend=dict(x=0.01, y=0.99)
)

fig.show()

# Trending Items over time

In [ ]:
date_item_df = pd.DataFrame(
    df.set_index('invoice_date').groupby([
        pd.Grouper(freq='ME'), 'stock_code'
    ])['quantity'].sum()
)
date_item_df

In [ ]:
# Rank items by the last month sales
last_month_sorted_df = date_item_df.loc['2011-11-30'].sort_values(
    by='quantity', ascending=False
).reset_index()

last_month_sorted_df

In [ ]:
# Regroup for top 5 items
date_item_df = pd.DataFrame(
    df.loc[
        df['stock_code'].isin([23084, 84826, 22197, 22086, '85099B'])
    ].set_index('invoice_date').groupby([
        pd.Grouper(freq='ME'), 'stock_code'
    ])['quantity'].sum()
)
date_item_df

In [ ]:
trending_itmes_df = date_item_df.reset_index().pivot(
	index='invoice_date',
	columns='stock_code',
	values='quantity'
).fillna(0)

trending_itmes_df

In [ ]:
fig = go.Figure()

for col in trending_itmes_df.columns:
    fig.add_trace(go.Scatter(
        x=trending_itmes_df.index,
        y=trending_itmes_df[col],
        mode='lines+markers',
        name=str(col)
    ))

fig.update_layout(
    title='Item Trends over Time',
    xaxis_title='date',
    yaxis_title='number of purchases',
    xaxis=dict(
        tickformat='%m.%Y',
        tickangle=45
    ),
    width=1000,
    height=700,
    template='ggplot2'
)

fig.show()